In [2]:
import numpy as np
import pandas as pd 
from tensorflow import keras

In [18]:


df = pd.read_csv(
    "household_power_consumption.txt",
    sep=";",
    usecols=["Date", "Time", "Global_active_power"],
    na_values=["?"],
    low_memory=True
)
df = df.dropna().reset_index(drop=True)
print(df.shape)
print(df.head())
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")
df.columns

(2049280, 3)
         Date      Time  Global_active_power
0  16/12/2006  17:24:00                4.216
1  16/12/2006  17:25:00                5.360
2  16/12/2006  17:26:00                5.374
3  16/12/2006  17:27:00                5.388
4  16/12/2006  17:28:00                3.666
240.28182792663574 MB


Index(['Date', 'Time', 'Global_active_power'], dtype='str')

In [19]:
from sklearn.preprocessing import MinMaxScaler
df["Global_active_power"]=df["Global_active_power"].astype(np.float32)
scaler = MinMaxScaler()

l = len(df)
x_train,x_valid,x_test= df[:int(l*0.7)],df[int(l*0.7):int(l*0.9)],df[int(l*0.9):]

In [20]:
x_train['Global_active_power'] = scaler.fit_transform(x_train[['Global_active_power']])
x_valid['Global_active_power'] = scaler.transform(x_valid[['Global_active_power']])
x_test['Global_active_power'] = scaler.transform(x_test[['Global_active_power']])

In [21]:
model=keras.models.Sequential([
    keras.layers.SimpleRNN(20,return_sequences=True,input_shape=[None,1]),
    keras.layers.SimpleRNN(20),
    keras.layers.Dense(10),
    
])

C:\Users\yukine\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [22]:
model.compile(loss="mse",optimizer="adam")

In [23]:
import numpy as np

n_steps = 24
horizon = 10

def create_sequences(data, n_steps, horizon):
    X, Y = [], []

    for i in range(len(data) - n_steps - horizon + 1):
        X.append(data[i : i + n_steps])
        Y.append(data[i + n_steps : i + n_steps + horizon])

    X = np.array(X, dtype=np.float32).reshape(-1, n_steps, 1)
    Y = np.array(Y, dtype=np.float32)

    return X, Y

In [24]:
train_values = x_train["Global_active_power"].to_numpy()
valid_values = x_valid["Global_active_power"].to_numpy()
test_values = x_test["Global_active_power"].to_numpy()
X_train, Y_train = create_sequences(train_values, n_steps, horizon)
X_valid, Y_valid = create_sequences(valid_values, n_steps, horizon)
X_test ,Y_test  =   create_sequences(test_values,n_steps,horizon)

In [26]:
history = model.fit(
    X_train,
    Y_train,
    epochs=5,
    batch_size=256,
    validation_data=(X_valid, Y_valid)
)

Epoch 1/5
5604/5604 ━━━━━━━━━━━━━━━━━━━━ 45s 8ms/step - loss: 0.0023 - val_loss: 0.0020
Epoch 2/5
5604/5604 ━━━━━━━━━━━━━━━━━━━━ 58s 10ms/step - loss: 0.0023 - val_loss: 0.0020
Epoch 3/5
5604/5604 ━━━━━━━━━━━━━━━━━━━━ 55s 10ms/step - loss: 0.0023 - val_loss: 0.0020
Epoch 4/5
5604/5604 ━━━━━━━━━━━━━━━━━━━━ 74s 8ms/step - loss: 0.0022 - val_loss: 0.0020
Epoch 5/5
5604/5604 ━━━━━━━━━━━━━━━━━━━━ 56s 10ms/step - loss: 0.0022 - val_loss: 0.0020


In [27]:
Y_pred = model.predict(X_test)
test_loss = model.evaluate(X_test, Y_test)
print("Test loss:", test_loss)

6403/6403 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step
6403/6403 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0015
Test loss: 0.0015499710571020842


X_train has NaN: True
Y_train has NaN: True
X_valid has NaN: True
Y_valid has NaN: True
X_test has NaN: True
Y_test has NaN: True
